# 2 — Preprocessing: Data Cleaning, Datasets, and DataLoaders

This notebook prepares the GTZAN dataset for model training. It is the combined work of two contributors:

- **Person A (Database):** connects to PostgreSQL to pull authoritative train/val/test split assignments, and verifies data quality (nulls, duplicates, corrupt-file exclusion)
- **Person B (Pipeline):** builds the `AudioDataset` class, audio helpers, augmentations, and PyTorch DataLoaders

**Output:** three ready-to-train DataLoaders (`train_loader`, `val_loader`, `test_loader`) that yield normalized mel spectrogram tensors and integer genre labels.

## 1. Imports and Configuration

The preprocessing pipeline uses `librosa` for audio loading and mel spectrogram extraction, then returns PyTorch tensors that can be passed directly into model training.

In [1]:
from pathlib import Path
import os

os.environ.setdefault("NUMBA_CACHE_DIR", str(Path.cwd() / ".numba-cache"))

import librosa
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


RANDOM_SEED = 42
SAMPLE_RATE = 22_050
CLIP_DURATION_SECONDS = 30
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
BATCH_SIZE = 16
FREQ_MASK_MAX_WIDTH = 16
EXCLUDED_FILES = {"jazz.00054.wav"}

FIXED_NUM_SAMPLES = SAMPLE_RATE * CLIP_DURATION_SECONDS

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Metadata DataFrame from Database

`audio_clips` stores one row per wav file with its genre label and train/val/test split assignment.
We pull all rows, then resolve the stored relative path to an absolute path so librosa can load each file.

In [ ]:
import os
import psycopg2
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "code").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the audio-genre-classifier repo root.")


repo_root = find_repo_root(Path.cwd())
load_dotenv(repo_root / ".env")

conn = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ.get("POSTGRES_PASSWORD", ""),
)

metadata_df = pd.read_sql(
    "SELECT file_path, label, split FROM audio_clips ORDER BY file_path",
    conn,
)
conn.close()

# file_path is stored relative to repo root — resolve to absolute for librosa
metadata_df["file_path"] = metadata_df["file_path"].apply(
    lambda p: str(repo_root / p)
)

metadata_df.head()

In [3]:
print(metadata_df.shape)
print(metadata_df["split"].value_counts())
print(metadata_df["label"].value_counts().sort_index())

(999, 3)
split
train    800
val      100
test      99
Name: count, dtype: int64
label
blues        100
classical    100
country      100
disco        100
hiphop       100
jazz          99
metal        100
pop          100
reggae       100
rock         100
Name: count, dtype: int64


## 3. Data Cleaning

Three checks before any model training:

1. **Null values** — the DB enforces `NOT NULL` on all columns, but we confirm here
2. **Duplicates** — each `file_path` is a primary key in the DB, so duplicates are impossible, but we verify the dataframe too
3. **Corrupt file exclusion** — `jazz.00054.wav` is a corrupt recording that fails to load. It was excluded when `audio_clips` was populated (`db/populate.py`), so it will never appear in the query results — confirmed below

In [ ]:
null_count = metadata_df.isnull().sum().sum()
dup_count = metadata_df.duplicated().sum()
corrupt_count = metadata_df["file_path"].str.contains("jazz.00054", regex=False).sum()

print(f"Null values:          {null_count}  (expect 0)")
print(f"Duplicate rows:       {dup_count}  (expect 0)")
print(f"jazz.00054 rows:      {corrupt_count}  (expect 0)")
print(f"\nTotal songs: {len(metadata_df)} — splits: {metadata_df['split'].value_counts().to_dict()}")

## 3. Label Encoding

PyTorch classification targets should be integer class IDs. The mapping is built from the dataframe labels so the same encoding can be reused during training and prediction.

In [4]:
labels = sorted(metadata_df["label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

label_to_idx

{'blues': 0,
 'classical': 1,
 'country': 2,
 'disco': 3,
 'hiphop': 4,
 'jazz': 5,
 'metal': 6,
 'pop': 7,
 'reggae': 8,
 'rock': 9}

## 4. Audio Helpers

Each clip is loaded at the same sample rate, then padded or truncated to a fixed number of samples. This keeps every mel spectrogram the same shape, which lets the DataLoader stack examples into batches.

In [5]:
def pad_or_truncate(audio: np.ndarray, target_num_samples: int) -> np.ndarray:
    """Return audio with exactly target_num_samples samples."""
    if len(audio) > target_num_samples:
        return audio[:target_num_samples]

    if len(audio) < target_num_samples:
        padding = target_num_samples - len(audio)
        return np.pad(audio, (0, padding), mode="constant")

    return audio


def augment_waveform(audio: np.ndarray) -> np.ndarray:
    """Apply light waveform augmentations for training examples only."""
    # Random gain changes volume without changing genre identity.
    gain = np.random.uniform(0.8, 1.2)
    audio = audio * gain

    # Small time shift makes the model less dependent on exact alignment.
    max_shift = int(0.1 * SAMPLE_RATE)
    shift = np.random.randint(-max_shift, max_shift + 1)
    audio = np.roll(audio, shift)

    # Low-amplitude noise can improve robustness.
    noise = np.random.normal(0, 0.005, size=audio.shape)
    audio = audio + noise

    return audio.astype(np.float32)


def audio_to_mel_spectrogram(audio: np.ndarray) -> np.ndarray:
    """Convert waveform audio into a normalized log-mel spectrogram."""
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalize each spectrogram to a stable range for model input.
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


def augment_mel_spectrogram(mel: np.ndarray) -> np.ndarray:
    """Apply light frequency masking for training spectrograms only."""
    mel = mel.copy()
    max_width = min(FREQ_MASK_MAX_WIDTH, mel.shape[0])
    mask_width = np.random.randint(0, max_width + 1)

    if mask_width > 0:
        start = np.random.randint(0, mel.shape[0] - mask_width + 1)
        mel[start:start + mask_width, :] = 0.0

    return mel.astype(np.float32)

## 5. PyTorch Dataset

`AudioDataset` accepts a dataframe with `file_path`, `label`, and `split`. Augmentation is only enabled for rows where `split == 'train'` and `augment=True`, so validation and test examples remain stable.

In [6]:
class AudioDataset(Dataset):
    """PyTorch dataset that loads wav files and returns mel spectrogram tensors."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        label_to_idx: dict[str, int],
        sample_rate: int = SAMPLE_RATE,
        fixed_num_samples: int = FIXED_NUM_SAMPLES,
        augment: bool = False,
    ):
        expected_columns = {"file_path", "label", "split"}
        missing_columns = expected_columns - set(dataframe.columns)
        if missing_columns:
            raise ValueError(f"Dataframe is missing columns: {sorted(missing_columns)}")

        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.sample_rate = sample_rate
        self.fixed_num_samples = fixed_num_samples
        self.augment = augment

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.dataframe.iloc[index]

        audio, _ = librosa.load(row["file_path"], sr=self.sample_rate, mono=True)
        audio = pad_or_truncate(audio, self.fixed_num_samples)

        if self.augment and row["split"] == "train":
            audio = augment_waveform(audio)

        mel = audio_to_mel_spectrogram(audio)

        if self.augment and row["split"] == "train":
            mel = augment_mel_spectrogram(mel)

        # Add channel dimension: [channels, n_mels, time_frames].
        features = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor(self.label_to_idx[row["label"]], dtype=torch.long)

        return features, target

## 6. Train, Validation, and Test DataLoaders

The training loader shuffles examples and uses augmentation. Validation and test loaders do not shuffle or augment so metrics are repeatable.

In [7]:
train_df = metadata_df[metadata_df["split"] == "train"].copy()
val_df = metadata_df[metadata_df["split"] == "val"].copy()
test_df = metadata_df[metadata_df["split"] == "test"].copy()

train_dataset = AudioDataset(train_df, label_to_idx=label_to_idx, augment=True)
val_dataset = AudioDataset(val_df, label_to_idx=label_to_idx, augment=False)
test_dataset = AudioDataset(test_df, label_to_idx=label_to_idx, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples  : {len(val_dataset)}")
print(f"Test examples : {len(test_dataset)}")

Train examples: 800
Val examples  : 100
Test examples : 99


## 7. Verification Cell

Run this cell before model training. It confirms that the DataLoader can produce a batch, that features and labels have the expected tensor types, and that spectrogram values are finite.

In [8]:
batch_features, batch_labels = next(iter(train_loader))

print(f"Batch feature shape : {batch_features.shape}")
print(f"Batch feature dtype  : {batch_features.dtype}")
print(f"Batch label shape   : {batch_labels.shape}")
print(f"Batch label dtype    : {batch_labels.dtype}")
print(f"Feature min value   : {batch_features.min().item():.4f}")
print(f"Feature max value   : {batch_features.max().item():.4f}")
print(f"All feature values finite: {torch.isfinite(batch_features).all().item()}")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batch feature shape : torch.Size([16, 1, 128, 1292])
Batch feature dtype  : torch.float32
Batch label shape   : torch.Size([16])
Batch label dtype    : torch.int64
Feature min value   : -3.2444
Feature max value   : 4.4979
All feature values finite: True


## Notes

The temporary `metadata_df` has been replaced with a live query against the `audio_clips` PostgreSQL table (see `db/schema.sql` and `db/populate.py`). Split assignments are now reproducible and consistent across all teammates — each person runs `db/populate.py` locally with the same random seed and gets identical splits.

The downstream code (label encoding, AudioDataset, DataLoaders) is unchanged.